# setup

In [1]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import cv2
import torch
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available())


PROJECT_ROOT: D:\hoiancomputervision\ticket_suspicion_project
CUDA: False


# Load config và frame index

In [2]:
# Đọc các file cấu hình từ thư mục configs để biết tham số chạy và các vùng ROI.
pipeline_cfg = load_yaml(PROJECT_ROOT / "configs" / "pipeline_config.yaml")

mot_root = PROJECT_ROOT / "data" / "raw" / "MOT17" / "train"

def parse_seqinfo(seq_dir):
    fps = 30.0
    width = 0
    height = 0
    ini_path = seq_dir / "seqinfo.ini"
    if ini_path.exists():
        import configparser
        config = configparser.ConfigParser()
        config.read(ini_path)
        if "Sequence" in config:
            seq_info = config["Sequence"]
            fps = float(seq_info.get("frameRate", seq_info.get("frame_rate", fps)))
            width = int(seq_info.get("imWidth", seq_info.get("imgWidth", width)))
            height = int(seq_info.get("imHeight", seq_info.get("imgHeight", height)))
    return fps, width, height


def build_frame_index(root_dir):
    rows = []
    if not root_dir.exists():
        return pd.DataFrame(rows)

    for seq_dir in sorted(root_dir.iterdir()):
        img_dir = seq_dir / "img1"
        if not img_dir.exists() or not img_dir.is_dir():
            continue

        sequence_name = seq_dir.name
        camera_id = sequence_name
        fps, width, height = parse_seqinfo(seq_dir)
        image_paths = sorted(img_dir.glob("*.jpg"))

        for frame_id, image_path in enumerate(image_paths, start=1):
            timestamp_sec = (frame_id - 1) / fps if fps > 0 else 0.0
            rows.append({
                "camera_id": camera_id,
                "sequence_name": sequence_name,
                "frame_id": frame_id,
                "timestamp_sec": timestamp_sec,
                "image_path": str(image_path),
                "width": width,
                "height": height,
                "fps": fps,
            })

    return pd.DataFrame(rows)

frame_index_df = build_frame_index(mot_root)
frame_index_path = PROJECT_ROOT / "outputs" / "frames" / "frame_index.csv"
frame_index_path.parent.mkdir(parents=True, exist_ok=True)
frame_index_df.to_csv(frame_index_path, index=False)
print("Loaded frame_index from:", mot_root)
print("Sequences found:", frame_index_df["sequence_name"].nunique())
print("Total frames:", len(frame_index_df))
frame_index_df.head()

Loaded frame_index from: D:\hoiancomputervision\ticket_suspicion_project\data\raw\MOT17\train
Sequences found: 7
Total frames: 5316


,camera_id,sequence_name,frame_id,timestamp_sec,image_path,width,height,fps
0,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.000000,D:\hoiancomputervision\ticket_suspicion_projec...,1920,1080,30.0
1,MOT17-02-FRCNN,MOT17-02-FRCNN,2,0.033333,D:\hoiancomputervision\ticket_suspicion_projec...,1920,1080,30.0
2,MOT17-02-FRCNN,MOT17-02-FRCNN,3,0.066667,D:\hoiancomputervision\ticket_suspicion_projec...,1920,1080,30.0
3,MOT17-02-FRCNN,MOT17-02-FRCNN,4,0.100000,D:\hoiancomputervision\ticket_suspicion_projec...,1920,1080,30.0
4,MOT17-02-FRCNN,MOT17-02-FRCNN,5,0.133333,D:\hoiancomputervision\ticket_suspicion_projec...,1920,1080,30.0


# Load YOLO

In [3]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
det_cfg = pipeline_cfg["detector"]

model = YOLO(det_cfg["model"])
device = 0 if torch.cuda.is_available() else "cpu"

print("Using device:", device)


Using device: cpu


# Chạy YOLO + ByteTrack

In [4]:
# Lưu dữ liệu trung gian dưới dạng CSV để các bước khác có thể đọc lại được.
conf_thres = det_cfg["confidence_threshold"]
iou_thres = det_cfg["iou_threshold"]
person_class_id = det_cfg["person_class_id"]
tracker_yaml = pipeline_cfg["tracker"]["tracker_yaml"]

output_rows = []

for camera_id, cam_df in frame_index_df.groupby("camera_id"):
    cam_df = cam_df.sort_values("frame_id").reset_index(drop=True)

    print(f"Processing camera: {camera_id}, frames: {len(cam_df)}")

    for _, frame_row in tqdm(cam_df.iterrows(), total=len(cam_df)):
        frame = cv2.imread(frame_row["image_path"])

        if frame is None:
            continue

        results = model.track(
            source=frame,
            persist=True,
            tracker=tracker_yaml,
            conf=conf_thres,
            iou=iou_thres,
            classes=[person_class_id],
            verbose=False,
            device=device
        )

        boxes = results[0].boxes

        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        ids = boxes.id.cpu().numpy().astype(int)

        for bbox, conf, track_id in zip(xyxy, confs, ids):
            x1, y1, x2, y2 = bbox.tolist()

            output_rows.append({
                "camera_id": camera_id,
                "sequence_name": frame_row["sequence_name"],
                "frame_id": int(frame_row["frame_id"]),
                "timestamp_sec": float(frame_row["timestamp_sec"]),
                "image_path": frame_row["image_path"],
                "local_track_id": int(track_id),
                "confidence": float(conf),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2)
            })

tracks_df = pd.DataFrame(output_rows)

out_path = PROJECT_ROOT / "outputs/tracks/tracks.csv"
save_csv(tracks_df, out_path)

print("Saved:", out_path)
print("Rows:", len(tracks_df))
tracks_df.head()


Processing camera: MOT17-02-FRCNN, frames: 600


  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 1/600 [00:01<18:23,  1.84s/it]

  1%|          | 4/600 [00:01<03:49,  2.60it/s]

  1%|          | 7/600 [00:02<01:59,  4.96it/s]

...

100%|█████████▉| 747/750 [00:35<00:00, 22.50it/s]

100%|██████████| 750/750 [00:35<00:00, 22.51it/s]

100%|██████████| 750/750 [00:35<00:00, 21.08it/s]

Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\tracks\tracks.csv
Rows: 45445


,camera_id,sequence_name,frame_id,timestamp_sec,image_path,local_track_id,confidence,x1,y1,x2,y2
0,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.0,D:\hoiancomputervision\ticket_suspicion_projec...,1,0.849230,1337.596191,418.733643,1499.459595,785.622070
1,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.0,D:\hoiancomputervision\ticket_suspicion_projec...,2,0.836228,583.730347,447.518890,674.571960,708.177368
2,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.0,D:\hoiancomputervision\ticket_suspicion_projec...,3,0.769477,1457.200195,434.877197,1595.463135,762.812866
3,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.0,D:\hoiancomputervision\ticket_suspicion_projec...,4,0.736684,439.472473,445.419098,550.020020,719.356812
4,MOT17-02-FRCNN,MOT17-02-FRCNN,1,0.0,D:\hoiancomputervision\ticket_suspicion_projec...,5,0.542474,732.278381,458.517822,769.481384,571.080322


# Visualization tracking

In [5]:
# Tạo ảnh minh họa bounding box và ID track để kiểm tra chất lượng tracking.
vis_dir = PROJECT_ROOT / "outputs/visualizations/tracking"
vis_dir.mkdir(parents=True, exist_ok=True)

sample_frames = sorted(tracks_df["frame_id"].unique())[::100][:10]

for fid in sample_frames:
    frame_rows = tracks_df[tracks_df["frame_id"] == fid]

    if len(frame_rows) == 0:
        continue

    img_path = frame_rows.iloc[0]["image_path"]
    frame = cv2.imread(img_path)

    for _, row in frame_rows.iterrows():
        x1, y1, x2, y2 = map(int, [row.x1, row.y1, row.x2, row.y2])
        tid = int(row.local_track_id)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"ID {tid}",
            (x1, max(0, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

    save_path = vis_dir / f"tracking_frame_{fid:06d}.jpg"
    cv2.imwrite(str(save_path), frame)

print("Saved visualization to:", vis_dir)


Saved visualization to: D:\hoiancomputervision\ticket_suspicion_project\outputs\visualizations\tracking


# Some check

In [6]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

candidate_files = list((PROJECT_ROOT / "outputs").rglob("*track*.csv"))

print("Track files found:")
for p in candidate_files:
    print(p)

for p in candidate_files:
    try:
        df = pd.read_csv(p)
        print("\nFILE:", p)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head())
    except Exception as e:
        print("Cannot read:", p, e)


Track files found:
D:\hoiancomputervision\ticket_suspicion_project\outputs\tracks\tracks.csv

FILE: D:\hoiancomputervision\ticket_suspicion_project\outputs\tracks\tracks.csv
Rows: 45445
Columns: ['camera_id', 'sequence_name', 'frame_id', 'timestamp_sec', 'image_path', 'local_track_id', 'confidence', 'x1', 'y1', 'x2', 'y2']
        camera_id   sequence_name  frame_id  timestamp_sec  \
0  MOT17-02-FRCNN  MOT17-02-FRCNN         1            0.0   
1  MOT17-02-FRCNN  MOT17-02-FRCNN         1            0.0   
2  MOT17-02-FRCNN  MOT17-02-FRCNN         1            0.0   
3  MOT17-02-FRCNN  MOT17-02-FRCNN         1            0.0   
4  MOT17-02-FRCNN  MOT17-02-FRCNN         1            0.0   

                                          image_path  local_track_id  \
0  D:\hoiancomputervision\ticket_suspicion_projec...               1   
1  D:\hoiancomputervision\ticket_suspicion_projec...               2   
2  D:\hoiancomputervision\ticket_suspicion_projec...               3   
3  D:\hoiancom